[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C41_Deep_RL_Course/04_model_based_world/04_model_based_world.ipynb)

# 04 · 基于模型与世界模型（纯 numpy，从零）

目标：从零学一个 **toy 动态模型**，用它做 **MPC 随机打靶** 把一个点质量控制到目标；实现 **Dyna**（真实+想象经验混合）；亲眼看 **模型误差如何累积**；并用一个 toy 展示 **latent 想象**（世界模型骨架）。

路线：学动态模型 → MPC 随机打靶规划 → Dyna 混合 → 复合误差 → latent 想象直觉 → ✏️ 练习(动态拟合/MPC打靶/Dyna/误差累积) → 📖 答案 → 🧪 真实采样效率胶囊。

> 心智模型：**学模型=稳的监督学习；MPC=每步在模型里搜、只执行第一步；Dyna=用想象假经验冒充真经验；误差随 rollout 步数指数累积**。

## 1 · 学一个动态模型（toy 点质量）

环境：1D 点质量，状态 `[pos, vel]`，动作 = 力 ∈ [-1,1]。目标到 `pos=0, vel=0`。
从随机交互数据学一个**线性动态模型** `s' = f(s,a)`（这个 toy 动态本就线性，最小二乘可精确拟合）。

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

def true_dynamics(s, a):
    pos, vel = s; a = np.clip(a, -1, 1)
    vel2 = vel + 0.1 * a
    pos2 = pos + 0.1 * vel2
    return np.array([pos2, vel2])

def reward_fn(s, a):
    return -(s[0]**2 + 0.1*s[1]**2 + 0.01*a**2)   # 越接近原点、越省力越好

# 收集随机交互数据，拟合线性模型 s' = [s, a, 1] @ W
X, Y = [], []
for _ in range(2000):
    s = rng.uniform(-2, 2, 2); a = rng.uniform(-1, 1)
    X.append([s[0], s[1], a, 1.0]); Y.append(true_dynamics(s, a))
X, Y = np.array(X), np.array(Y)
W, *_ = np.linalg.lstsq(X, Y, rcond=None)

def model_dynamics(s, a):
    return np.array([s[0], s[1], a, 1.0]) @ W

mse = ((X @ W - Y)**2).mean()
print(f'动态模型拟合 MSE = {mse:.2e}')
# 抽查模型预测 vs 真实
s_test = np.array([1.0, 0.5])
print('模型预测 s\' =', np.round(model_dynamics(s_test, 0.5), 4))
print('真实   s\' =', np.round(true_dynamics(s_test, 0.5), 4))
assert mse < 1e-6, '线性动态应被最小二乘精确拟合'
assert np.allclose(model_dynamics(s_test, 0.5), true_dynamics(s_test, 0.5), atol=1e-4)
print('✅ 动态模型学好了（学模型是稳的监督学习，无致命三要素）')

## 2 · MPC 随机打靶：用模型规划

每一步：在**模型**里撒 `n_samples` 条随机动作序列(长 `horizon`)，算每条的模型预测回报，选最优序列的**第一个动作**执行，下一步重新规划（滚动时域）。

In [ ]:
def mpc_action(s, horizon=15, n_samples=200, seed=0):
    r = np.random.default_rng(seed)
    best_ret, best_a0 = -1e18, 0.0
    for _ in range(n_samples):
        seq = r.uniform(-1, 1, horizon)        # 一条候选动作序列
        sim = s.copy(); ret = 0.0
        for a in seq:                          # 在模型里 rollout
            ret += reward_fn(sim, a)
            sim = model_dynamics(sim, a)
        if ret > best_ret:
            best_ret, best_a0 = ret, seq[0]    # 只记第一个动作
    return best_a0

# 在真实环境里用 MPC 控制
s = np.array([1.5, 0.0]); traj = [s[0]]
for t in range(40):
    a = mpc_action(s, seed=t)
    s = true_dynamics(s, a); traj.append(s[0])
print('MPC 控制下 pos 轨迹(每5步):', np.round(traj[::5], 3))
print(f'MPC 最终 |pos| = {abs(s[0]):.3f}')
# 对比随机动作
s_rand = np.array([1.5, 0.0])
for t in range(40): s_rand = true_dynamics(s_rand, rng.uniform(-1,1))
print(f'随机动作最终 |pos| = {abs(s_rand[0]):.3f}')
assert abs(s[0]) < 0.5, 'MPC 应把点质量控制到目标附近'
assert abs(s[0]) < abs(s_rand[0]) - 0.5, 'MPC 应远好于随机动作'
print('✅ 学到的模型 + MPC 随机打靶把点质量从 1.5 控制到 ~0（只执行第一步+重规划）')

## 3 · Dyna：真实 + 想象经验混合

用回 GridWorld（离散 Q-learning）。Dyna：每步真实交互后，① 更新 Q；② 更新模型；③ 从模型采 `k` 条想象经验**也更新 Q**。

对比 `k=0`（纯无模型）与 `k>0`（Dyna）的学习速度——Dyna 应**更快**学好。

In [ ]:
class GridWorld:
    def __init__(self, n=5): self.n=n; self.nS=n*n; self.nA=4; self.goal=n*n-1; self.s=0
    def reset(self): self.s=0; return self.s
    def step(self, a):
        r,c = self.s//self.n, self.s%self.n
        if a==0: r-=1
        elif a==1: r+=1
        elif a==2: c-=1
        elif a==3: c+=1
        r=min(max(r,0),self.n-1); c=min(max(c,0),self.n-1)
        self.s=r*self.n+c; done=(self.s==self.goal)
        return self.s, (1.0 if done else 0.0), done

def dyna_q(k_planning=0, episodes=40, seed=0, alpha=0.5, gamma=0.95, eps=0.2):
    '''k_planning=0 -> 纯 Q-learning; k>0 -> Dyna(每步额外 k 条想象更新)。'''
    r = np.random.default_rng(seed)
    env = GridWorld(5); Q = np.zeros((env.nS, env.nA))
    model = {}                                  # (s,a) -> (r, s') 确定性模型
    seen = []                                   # 见过的 (s,a)
    steps_to_goal = []
    for ep in range(episodes):
        s = env.reset(); steps = 0
        for _ in range(200):
            qs = Q[s]
            # ε-贪婪，且 argmax 随机打破平局(否则全 0 时永远选动作0卡住)
            a = int(r.integers(0,4)) if r.random()<eps else int(r.choice(np.flatnonzero(qs==qs.max())))
            s2, rew, done = env.step(a); steps += 1
            Q[s,a] += alpha*(rew + gamma*(1-done)*Q[s2].max() - Q[s,a])  # ① 直接 RL
            model[(s,a)] = (rew, s2)                                     # ② 学模型
            if (s,a) not in seen: seen.append((s,a))
            for _ in range(k_planning):                                 # ③ 想象更新
                ps, pa = seen[r.integers(0, len(seen))]
                pr, ps2 = model[(ps,pa)]
                pdone = (ps2 == env.goal)
                Q[ps,pa] += alpha*(pr + gamma*(1-pdone)*Q[ps2].max() - Q[ps,pa])
            s = s2
            if done: break
        steps_to_goal.append(steps)
    return steps_to_goal

no_model = dyna_q(k_planning=0, episodes=40, seed=0)
dyna     = dyna_q(k_planning=20, episodes=40, seed=0)
# 比较达到目标的总步数(越少=越快学会)。Dyna 用想象经验更快把价值传播开
print(f'纯Q-learning 40回合总步数 = {sum(no_model)}  (前10回合均 {np.mean(no_model[:10]):.0f})')
print(f'Dyna(k=20)   40回合总步数 = {sum(dyna)}  (前10回合均 {np.mean(dyna[:10]):.0f})')
assert sum(dyna) < sum(no_model), 'Dyna 总步数应更少(更快学会)'
print('✅ Dyna 用想象经验加速学习：同样真实交互，更快把价值传播、更快找到目标')

## 4 · 复合误差：模型误差随 rollout 累积

给模型加一点噪声（模拟真实模型的不完美），对比**真实轨迹**与**模型 rollout 轨迹**的偏差如何随步数增长。

结论：单步误差虽小，多步 rollout 会让它**累积放大** —— 这就是为什么 MPC/Dyna 都要 rollout 短。

In [ ]:
def noisy_model(s, a, r, noise=0.03):
    return model_dynamics(s, a) + r.normal(0, noise, 2)

r = np.random.default_rng(1)
s0 = np.array([1.0, 0.5]); acts = r.uniform(-1, 1, 20)
# 真实 rollout
strue = s0.copy(); true_traj = [strue.copy()]
for a in acts:
    strue = true_dynamics(strue, a); true_traj.append(strue.copy())
# 模型 rollout（误差喂回下一步）
smod = s0.copy(); errs = []
for i, a in enumerate(acts):
    smod = noisy_model(smod, a, r)
    errs.append(np.linalg.norm(smod - true_traj[i+1]))
print('模型 rollout 误差 @步 1/5/10/20:', [round(errs[i],4) for i in [0,4,9,19]])
# 误差总体趋势应增长(累积)
assert np.mean(errs[10:]) > np.mean(errs[:5]), '多步误差应大于少步(累积)'
assert errs[-1] > errs[0], '末步误差应大于首步'
print('✅ 复合误差：单步误差被反复喂回 -> 随 rollout 步数累积放大')
print('   -> 黄金法则：用模型走短(MPC只信第一步、MBPO只rollout几步)、用真实校准')

## 5 · 世界模型骨架：latent 编码 + latent 想象

世界模型在**压缩的 latent 空间**预测演化（比像素空间更稳）。
toy 版：把 2D 观测线性编码到 1D latent，在 latent 里学动态，再解码——展示「编码→latent预测→解码」的骨架。

In [ ]:
# toy 世界模型：观测 o in R^2，编码到 latent z in R^1，latent 里预测演化
# 真实生成过程：o = [z, z^2*0+z*0.5]，latent 演化 z' = 0.9*z + 0.1*a
def gen_obs(z): return np.array([z, 0.5*z])         # 观测是 latent 的函数
def true_latent_step(z, a): return 0.9*z + 0.1*a

# 收集 (o, a, o') 学：编码器 enc(o)->z, latent 动态 dyn(z,a)->z'
r = np.random.default_rng(0)
OZ = []   # 学编码器：从 o 回归 z（这里 z = o[0]，编码器学到取第一维）
for _ in range(500):
    z = r.uniform(-2, 2); OZ.append((gen_obs(z), z))
Oarr = np.array([o for o,_ in OZ]); Zarr = np.array([z for _,z in OZ])
enc_w, *_ = np.linalg.lstsq(Oarr, Zarr, rcond=None)   # z ≈ o @ enc_w
def encode(o): return o @ enc_w

# 学 latent 动态 z' = [z,a,1]@dyn_w
ZA, Z2 = [], []
for _ in range(500):
    z = r.uniform(-2,2); a = r.uniform(-1,1)
    ZA.append([z, a, 1.0]); Z2.append(true_latent_step(z, a))
dyn_w, *_ = np.linalg.lstsq(np.array(ZA), np.array(Z2), rcond=None)
def latent_step(z, a): return np.array([z, a, 1.0]) @ dyn_w

# 在 latent 里『想象』一条 rollout，再解码回观测，对比真实
z0 = encode(gen_obs(1.0)); acts = [0.5, -0.3, 0.2, -0.1]
z_imag = z0; z_true = 1.0
for a in acts:
    z_imag = latent_step(z_imag, a); z_true = true_latent_step(z_true, a)
print(f'latent 想象终点 z={float(z_imag):.3f}, 真实 z={z_true:.3f}')
print(f'解码回观测: 想象={np.round(gen_obs(float(z_imag)),3)}, 真实={np.round(gen_obs(z_true),3)}')
assert abs(encode(gen_obs(1.0)) - 1.0) < 1e-3, '编码器应学到从观测恢复 latent'
assert abs(float(z_imag) - z_true) < 0.05, 'latent 想象应贴近真实演化'
print('✅ 世界模型骨架：编码→latent想象→解码（Dreamer 在此之上用 RNN+actor-critic 做大）')

---
## ✏️ 练习 1：动态模型拟合误差

实现 `fit_dynamics(X, Y)`：用最小二乘拟合 `Y = X @ W`，返回 `(W, mse)`。
`X` 是 `(n, d)` 的特征 `[s..., a, 1]`，`Y` 是 `(n, state_dim)` 的下一状态。

In [ ]:
def fit_dynamics(X, Y):
    # TODO: 用 np.linalg.lstsq 解 W；mse = mean((X@W - Y)^2)。返回 (W, mse)
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
rng = np.random.default_rng(2)
Xt = np.column_stack([rng.uniform(-1,1,(200,3)), np.ones(200)])
W_true_ex = rng.standard_normal((4, 2))
Yt = Xt @ W_true_ex
W_ex, mse = fit_dynamics(Xt, Yt)         # 用独立变量名，勿覆盖全局动态模型的 W
assert mse < 1e-10, '无噪声线性数据应被精确拟合'
assert np.allclose(W_ex, W_true_ex, atol=1e-6), '应恢复真实 W'
print(f'拟合 MSE={mse:.2e}, 恢复 W 正确')
print('✅ 练习 1 通过：最小二乘拟合动态模型')

## ✏️ 练习 2：MPC 随机打靶（评估一条序列）

实现 `rollout_return(s, action_seq, model, reward_fn)`：在 `model` 里从 `s` 执行 `action_seq`，返回累计回报（先算 reward 再推进状态）。这是 MPC 打靶的核心子程序。

In [ ]:
def rollout_return(s, action_seq, model, reward_fn):
    # TODO: sim=s.copy(); ret=0; 对每个 a: ret+=reward_fn(sim,a); sim=model(sim,a)。返回 ret
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
s = np.array([1.0, 0.0])
# 一条把点推向原点的序列 应比 一条推离的序列 回报更高
toward = np.array([-1.0, -1.0, -1.0])      # 负力使 vel<0, pos 减小
away = np.array([1.0, 1.0, 1.0])
r_toward = rollout_return(s, toward, model_dynamics, reward_fn)
r_away = rollout_return(s, away, model_dynamics, reward_fn)
assert r_toward > r_away, '推向原点的序列回报应更高'
print(f'推向原点回报={r_toward:.3f} > 推离回报={r_away:.3f}')
print('✅ 练习 2 通过：模型内 rollout 评估动作序列回报')

## ✏️ 练习 3：Dyna 的想象更新

实现 `imagination_update(Q, model, seen, rng, alpha, gamma, goal)`：从 `seen` 里随机挑一个 `(s,a)`，用 `model[(s,a)]=(r,s')` 做**一次** Q 更新（想象经验）。返回更新后的 Q（原地改也可）。

In [ ]:
def imagination_update(Q, model, seen, rng, alpha=0.5, gamma=0.95, goal=24):
    # TODO: 随机选 (s,a); (r,s2)=model[(s,a)]; done=(s2==goal);
    #       Q[s,a] += alpha*(r + gamma*(1-done)*Q[s2].max() - Q[s,a]); 返回 Q
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
Q = np.zeros((25, 4))
model = {(23, 3): (1.0, 24), (10, 1): (0.0, 15)}   # (23,右)->目标 得 1
seen = [(23, 3), (10, 1)]
rg = np.random.default_rng(0)
# 反复想象更新，(23,3) 的 Q 应被抬起来(它直达目标 r=1)
for _ in range(200): imagination_update(Q, model, seen, rg, goal=24)
assert Q[23, 3] > 0.9, '直达目标的 (23,右) 经想象更新 Q 应接近 1'
print(f'想象更新后 Q[23,右]={Q[23,3]:.3f} (直达目标)')
print('✅ 练习 3 通过：Dyna 用模型生成的想象经验更新 Q')

## ✏️ 练习 4：复合误差度量

实现 `rollout_errors(s0, acts, true_fn, model_fn)`：分别用真实与模型从 `s0` rollout `acts`，返回每一步**模型状态与真实状态的欧氏距离**列表（长度 = len(acts)）。

In [ ]:
def rollout_errors(s0, acts, true_fn, model_fn):
    # TODO: 并行推进 strue 和 smod；每步记录 norm(smod - strue)。返回误差列表
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
rg = np.random.default_rng(3)
def biased_model(s, a): return model_dynamics(s, a) + np.array([0.02, 0.0])  # 有系统偏差
errs = rollout_errors(np.array([0.5, 0.0]), [0.3]*10, true_dynamics, biased_model)
assert len(errs) == 10
assert errs[-1] > errs[0], '有偏模型的误差应随步数累积增长'
assert all(e >= 0 for e in errs)
print('误差序列:', [round(e,4) for e in errs])
print('✅ 练习 4 通过：度量模型 rollout 的复合误差')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1
def fit_dynamics(X, Y):
    W, *_ = np.linalg.lstsq(X, Y, rcond=None)
    mse = ((X @ W - Y) ** 2).mean()
    return W, mse

In [ ]:
# 练习 2
def rollout_return(s, action_seq, model, reward_fn):
    sim = s.copy(); ret = 0.0
    for a in action_seq:
        ret += reward_fn(sim, a)
        sim = model(sim, a)
    return ret

In [ ]:
# 练习 3
def imagination_update(Q, model, seen, rng, alpha=0.5, gamma=0.95, goal=24):
    s, a = seen[rng.integers(0, len(seen))]
    r, s2 = model[(s, a)]
    done = (s2 == goal)
    Q[s, a] += alpha * (r + gamma * (1 - done) * Q[s2].max() - Q[s, a])
    return Q

In [ ]:
# 练习 4
def rollout_errors(s0, acts, true_fn, model_fn):
    strue = s0.copy(); smod = s0.copy(); errs = []
    for a in acts:
        strue = true_fn(strue, a); smod = model_fn(smod, a)
        errs.append(float(np.linalg.norm(smod - strue)))
    return errs

---
## 🧪 真实数据胶囊：基于模型的采样效率优势

下面是几类方法在连续控制（MuJoCo 类）上达到某性能所需的**真实交互步数**量级（约数，来自公开论文）。基于模型方法的样本效率优势一目了然。

In [ ]:
# 达到良好性能所需真实环境步数(量级；越小越省样本)
SAMPLE_EFFICIENCY = {
    'DQN (Atari, model-free)':      200_000_000,   # 2亿帧
    'SAC (MuJoCo, model-free)':       1_000_000,   # ~百万步
    'MBPO (model-based + SAC)':         100_000,   # ~十万步
    'PETS (model-based MPC)':           50_000,    # ~万级
    'Dreamer (world model)':           100_000,    # ~十万步(且像素输入)
}
print(f"{'方法':38s}{'真实步数':>14s}")
for k, v in SAMPLE_EFFICIENCY.items():
    print(f'{k:38s}{v:>14,}')
# 基于模型(MBPO/PETS/Dreamer)比同任务的 model-free(SAC) 省约 10x 真实样本
ratio = SAMPLE_EFFICIENCY['SAC (MuJoCo, model-free)'] / SAMPLE_EFFICIENCY['MBPO (model-based + SAC)']
print(f'\nMBPO 比 SAC 省约 {ratio:.0f}x 真实交互(用模型想象补足)')
assert ratio >= 5, '基于模型应显著更省样本'
print('✅ 模型换样本效率：真实交互昂贵时，基于模型是关键武器')

**🧪 胶囊练习**：实现 `sample_efficiency_gain(model_free_steps, model_based_steps)`：返回基于模型相对无模型节省的真实样本倍数 `model_free / model_based`。

In [ ]:
def sample_efficiency_gain(model_free_steps, model_based_steps):
    # TODO: 返回 model_free_steps / model_based_steps
    raise NotImplementedError

In [ ]:
# 自测
g = sample_efficiency_gain(SAMPLE_EFFICIENCY['SAC (MuJoCo, model-free)'],
                           SAMPLE_EFFICIENCY['PETS (model-based MPC)'])
assert g == 20.0
print(f'PETS 比 SAC 省 {g:.0f}x 真实样本 ✅ 胶囊练习通过')

In [ ]:
# 📖 胶囊参考答案
def sample_efficiency_gain(model_free_steps, model_based_steps):
    return model_free_steps / model_based_steps

### 小结
- **基于模型 RL**：先学动态模型 `ŝ'=f(s,a)`(稳的监督学习)，再用它**规划**或**生成想象经验**，用模型换**样本效率**。
- **MPC 随机打靶**：每步在模型里撒随机动作序列、选最优序列的**第一个动作**执行、重规划。只信第一步 -> 对模型误差鲁棒。
- **Dyna**：每步真实交互后，用模型采 k 条**想象经验**也更新 Q —— 对学习算法『假经验冒充真经验』，省真实样本。
- **复合误差**：单步模型误差被反复喂回 -> 随 rollout 步数**指数累积**。黄金法则：**用模型走短、用真实校准**。
- **世界模型**(Ha & Schmidhuber / Dreamer)：在**latent 空间**编码→想象→解码，在『梦里』训练策略，是基于模型的极致。

你已从零学模型、做 MPC 规划、实现 Dyna、看清复合误差。下一站：**模块 05 · 探索与前沿** —— 稀疏奖励下如何系统探索、多智能体、以及把 RL 变成序列建模(Decision Transformer)。